In [1]:
import os
# 只使用第一个 GPU (索引为 0)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed,
)

from datasets import load_dataset

seed = 42
# model_name_or_path = "google-bert/bert-base-chinese"
# model_name_or_path = "./models/google-bert-chinese"
# model_name_or_path = "./models/bert-large"
model_name_or_path = "../models/chinese-roberta-wwm-ext-large"
# model_name_or_path = "/Users/xuzhiguo/workspace/python/lx/bert_classify/models/chinese-modernbert-large-wwm"
output_dir = "output"
max_length = 512
num_labels = 20
lr = 2e-5
batch_size = 16
eval_batch_size = 16
epochs = 12
fp16 = True
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

set_seed(seed)

train_file = "data/train8.csv"
valid_file = "data/train8.csv"
raw_datasets = load_dataset("csv", data_files={"train": train_file, "validation": valid_file})

/home/zhiguo/miniconda3/envs/bert_intent_classify/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
tokenizer = AutoTokenizer.from_pretrained(model_name_or_path, use_fast=True)


def tokenize_function(examples):
    # examples["text"] can be a list
    return tokenizer(examples["text"], truncation=True, max_length=max_length, padding=True, return_tensors="pt")


tokenized = raw_datasets.map(
    tokenize_function,
    batched=True,
    batch_size=8,
    remove_columns=[c for c in raw_datasets["train"].column_names if c not in ("text", "label")],
)

#

In [3]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name_or_path, num_labels=num_labels
)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 46578.23it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: ./models/chinese-roberta-wwm-ext-large
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.

In [4]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=lr,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=eval_batch_size,
    num_train_epochs=epochs,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    fp16=False,
    bf16=False,
    save_total_limit=3,
    push_to_hub=False
)

from sklearn.metrics import accuracy_score, f1_score


def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    f1 = f1_score(labels, preds, average="weighted")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1}


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    # tokenizer=tokenizer,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,2.115656,0.299919,0.974719,0.974677
2,0.418786,0.068193,0.987828,0.987817
3,0.037156,0.009578,0.998127,0.998116
4,0.012490,0.003011,1.000000,1.000000
5,0.006277,0.002146,1.000000,1.000000
6,0.004041,0.001714,1.000000,1.000000
7,0.003492,0.001457,1.000000,1.000000
8,0.003209,0.001289,1.000000,1.000000
9,0.002675,0.001176,1.000000,1.000000
10,0.002506,0.001103,1.000000,1.000000


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.12s/it]


TrainOutput(global_step=804, training_loss=0.16868929817591474, metrics={'train_runtime': 113.5408, 'train_samples_per_second': 112.876, 'train_steps_per_second': 7.081, 'total_flos': 359696416010976.0, 'train_loss': 0.16868929817591474, 'epoch': 12.0})

In [7]:
from torch.nn.functional import cross_entropy
import torch

def forward_pass_with_label(batch):  # Place all input tensors on the same device as the model
    inputs = {k: v.to(device) for k, v in batch.items() if k in tokenizer.model_input_names}

    with torch.no_grad():
        output = model(**inputs)
        pred_label = torch.argmax(output.logits, axis=-1)
        loss = cross_entropy(output.logits, batch["label"].to(device), reduction="none")
        return {"loss": loss.cpu().numpy(),
                "predicted_label": pred_label.cpu().numpy(),
                "logits": output.logits,
                'prob': torch.softmax(output.logits, dim=-1)}


# Convert our dataset back to PyTorch tensors
tokenized.set_format("torch", columns=["input_ids", "attention_mask", "label"])  # Compute loss values
tokenized["validation"] = tokenized["validation"].map(forward_pass_with_label, batched=True, batch_size=1)

tokenized.set_format("pandas")
cols = ["text", "label", "predicted_label", "loss", "prob", "logits"]
df_test = tokenized["validation"][:][cols]
# df_test["label"] = df_test["label"].apply(index2labelF)
# df_test["predicted_label"] = df_test["predicted_label"].apply(index2labelF)
df_test.sort_values("loss", ascending=False)

Map: 100%|██████████| 1068/1068 [00:09<00:00, 112.68 examples/s]


,text,label,predicted_label,loss,prob,logits
179,我想一个人待会,3,3,0.010284,"[0.0003092593, 0.0002751811, 0.0007397286, 0.9...","[-0.4554612, -0.57221174, 0.4166422, 7.615586,..."
189,不再需要,3,3,0.008254,"[0.0002431108, 0.00011751317, 0.0007492126, 0....","[-0.36809224, -1.0950589, 0.75741345, 7.945647..."
164,关闭语音助手,3,3,0.005890,"[0.00015200087, 0.00016470831, 0.00048506938, ...","[-0.7341951, -0.65390503, 0.4262107, 8.051539,..."
172,停止服务,3,3,0.005153,"[0.00016025784, 0.00011021705, 0.0007893996, 0...","[-0.531879, -0.9062114, 1.0626092, 8.2016945, ..."
117,机器人，到地上去,2,2,0.004985,"[0.0004004094, 0.00010543906, 0.99502736, 0.00...","[0.24016401, -1.0941902, 8.058202, 0.3058646, ..."
...,...,...,...,...,...,...
492,你会打游戏吗？,8,8,0.001252,"[7.809417e-05, 4.809099e-05, 5.860932e-05, 0.0...","[-0.19616799, -0.6809884, -0.4831897, 0.106268..."
444,你会做数学题吗？,8,8,0.001250,"[8.071677e-05, 4.518587e-05, 5.801189e-05, 9.1...","[-0.16051492, -0.74067664, -0.49081373, -0.033..."
480,你会背唐诗吗？,8,8,0.001248,"[6.5854256e-05, 4.7060144e-05, 5.7821024e-05, ...","[-0.35630336, -0.6923209, -0.48639512, 0.00916..."
496,你有没有兄弟姐妹？,8,8,0.001244,"[6.91538e-05, 4.8327944e-05, 6.258897e-05, 0.0...","[-0.29433292, -0.6526556, -0.39407668, 0.12537..."


In [8]:
texts = [
    "循环巡视",
    "再见",
    "待在那别动",
    "向右原地转向",
    "停一停",
    "随便一句话",
    "下单吧",
    "确定",
    "好的",
    "我要奶茶",
    "奶茶",
    "点个生椰拿铁",
    "我要取个号",
    "怎么了",
    "好奇怪啊",
    ""
]

with open("data/other2.txt", "r", encoding="utf-8") as f:
    for line in f:
        texts.append(line.strip())

# 3. 分词编码，得到模型输入
inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
inputs.to(device)

first = True
# 4. 前向推理
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits  # 形状: (batch_size, num_labels)
    data = torch.softmax(logits, dim=-1)
    max_index = torch.argmax(data, dim=1)
    for i in range(len(max_index)):
        index = max_index[i].item()
        prob = data[i, index]
        if index != 8:
            print(prob.item(), texts[i], index)


0.9972776770591736 循环巡视 9
0.9963307976722717 再见 3
0.9955798983573914 待在那别动 15
0.9970263838768005 向右原地转向 11
0.9969602227210999 停一停 15
0.972032368183136 怎么了 3


In [11]:
texts = []
with open("data/train8.csv", "r", encoding="utf-8") as f:
    for line in f:
        text, label = line.strip().split(',')
        texts.append(text)
texts = texts[1:]

inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt")
inputs.to(device)

with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits  # 形状: (batch_size, num_labels)
    data = torch.softmax(logits, dim=-1)
    max_index = torch.argmax(data, dim=1)

    count = 0
    count2 = 0
    for i in range(len(max_index)):
        index = max_index[i].item()
        prob = data[i, index]
        if prob < 0.996:
            count += 1
            if index == 8:
                count2 += 1
            print(prob.item(), texts[i], index)
    print(count, count2)

0.9955915212631226 回到底座上去 1
0.9958606362342834 去底座上待着 1
0.9950273633003235 机器人，到地上去 2
0.9951199889183044 到地上去站好 2
0.9959946870803833 就到这里吧 3
0.9958187937736511 明天见 3
0.9941271543502808 关闭语音助手 3
0.9955684542655945 暂时不需要帮忙 3
0.9948605895042419 停止服务 3
0.995568573474884 就这样 3
0.9897691011428833 我想一个人待会 3
0.9954511523246765 别嘟囔了 3
0.9917804598808289 不再需要 3
0.9958231449127197 不要再说了 3
0.9956647753715515 别响了 3
0.9959200620651245 保持在我的后方 4
0.9957203269004822 继续走不要停 7
0.9958348274230957 保持当前方向前进 7
0.9958149790763855 继续走你的 7
0.9953731894493103 去到处转转 9
0.9956942796707153 机器人，去走两圈 9
0.9958699345588684 左转身 10
0.9959993362426758 原地左向转弯 10
0.9957439303398132 到左侧位置待命 12
0.9958869814872742 到右侧位置待命 13
0.9953386783599854 不要继续了 15
0.9957296252250671 到当前位置站好 15
0.995707094669342 取消锁定目标 16
0.9955108165740967 左转 18
0.9957857728004456 右转 19
30 0


In [ ]:
type(tokenized["train"])
tokenized.set_format("torch", columns=["input_ids", "attention_mask", "label", "text"])
for item in tokenized["train"]:
    if item["text"] == "向右方滑行":
        print(item)
        # input_ids = torch.tensor(item["input_ids"])

ids = [[101, 1403, 1381, 3175, 3998, 6121, 102]]
ids = [[101, 1403, 1381, 3175, 3998, 6121, 102, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]


def test():
    input_ids = [[101, 2518, 1381, 6804, 904, 3998, 102],
           [101, 1403, 1381, 904, 3566, 4919, 102],
           [101, 3308, 1381, 6804, 2918, 1220, 102],
           [101, 1403, 1381, 3175, 3998, 6121, 102],
           [101, 2518, 1381, 6804, 4919, 855, 102],
           [101, 2518, 1381, 3175, 904, 3998, 102],
           [101, 2518, 1381, 3175, 2398, 1220, 102],
           [101, 1403, 1381, 3175, 2544, 4919, 102]]
    attention_mask = [[1, 1, 1, 1, 1, 1, 1],
                      [1, 1, 1, 1, 1, 1, 1],
                      [1, 1, 1, 1, 1, 1, 1],
                      [1, 1, 1, 1, 1, 1, 1],
                      [1, 1, 1, 1, 1, 1, 1],
                      [1, 1, 1, 1, 1, 1, 1],
                      [1, 1, 1, 1, 1, 1, 1],
                      [1, 1, 1, 1, 1, 1, 1]]
    # ids = [[101, 1403, 1381, 3175, 3998, 6121, 102]]
    input_ids = torch.tensor(input_ids).to(device)
    attention_mask = torch.tensor(attention_mask).to(device)
    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    probs = torch.softmax(logits, dim=-1)
    print(torch.argmax(probs, dim=1))

with torch.no_grad():
    test()